# Visualizing CNN Kernels, Feature Maps, and "Dreams"

In this advanced tutorial, we will inspect and explore the inner representation of a Convolutional Neural Network (**LeNet-5**) trained on MNIST digits using **Nanograd**.

Specifically, we will:
1. **Train LeNet-5** on a small subset of MNIST for a few epochs.
2. **Visualize CNN Filters (Kernels)**: Plot the learned weights of the first convolutional layer (`conv1`) to see what kind of features they detect.
3. **Visualize Feature Maps**: Feed a test digit image through the network and plot the intermediate activations (feature maps) at each layer, revealing how different filters highlight edges, textures, or shapes.
4. **Class Activation Maximization ("Dreams")**: Start from a random noise image and use **gradient ascent on the pixels** to synthesize an input image that maximizes the network's prediction for a chosen digit class, showing us what the network "dreams" of when it thinks of a class.

---

In [ ]:
# Enable importing nanograd from parent directory
import sys
import os
sys.path.append(os.path.abspath('..'))

import gzip
import numpy as np
import matplotlib.pyplot as plt

from nanograd import Tensor, relu
from nanograd.nn import Conv2D, MaxPool2D, Flatten, Layer
from nanograd.optim import Adam
from nanograd.loss import SoftmaxCrossEntropy

np.random.seed(42)

## 1. Preparing and Training a Baseline LeNet-5 Model

We download and load MNIST, then train a LeNet-5 model for 5 epochs on 512 training images. This takes only ~3 seconds on CPU!

In [ ]:
def load_mnist_images(filename):
    with gzip.open(filename, 'rb') as f:
        magic, num, rows, cols = np.frombuffer(f.read(16), dtype=np.dtype('>i4'))
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.reshape(num, 1, rows, cols).astype(np.float32) / 255.0

def load_mnist_labels(filename):
    with gzip.open(filename, 'rb') as f:
        magic, num = np.frombuffer(f.read(8), dtype=np.dtype('>i4'))
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.astype(np.int64)

X_train_all = load_mnist_images('data/train-images-idx3-ubyte.gz')
y_train_all = load_mnist_labels('data/train-labels-idx1-ubyte.gz')
X_test_all = load_mnist_images('data/t10k-images-idx3-ubyte.gz')
y_test_all = load_mnist_labels('data/t10k-labels-idx1-ubyte.gz')

train_size = 512
test_size = 128

X_train, y_train_raw = X_train_all[:train_size], y_train_all[:train_size]
X_test, y_test_raw = X_test_all[:test_size], y_test_all[:test_size]

def to_one_hot(labels, num_classes=10):
    one_hot = np.zeros((labels.shape[0], num_classes))
    one_hot[np.arange(labels.shape[0]), labels] = 1.0
    return one_hot

y_train_oh = to_one_hot(y_train_raw)

class LeNet5:
    def __init__(self):
        self.conv1 = Conv2D(in_channels=1, num_filters=6, kernel_size=(5, 5), padding=2)
        self.pool1 = MaxPool2D(kernel_size=(2, 2), stride=2)
        self.conv2 = Conv2D(in_channels=6, num_filters=16, kernel_size=(5, 5), padding=0)
        self.pool2 = MaxPool2D(kernel_size=(2, 2), stride=2)
        self.flatten = Flatten()
        self.fc1 = Layer(num_neurons=120, num_inputs=400)
        self.fc2 = Layer(num_neurons=84, num_inputs=120)
        self.fc3 = Layer(num_neurons=10, num_inputs=84, activation_function=lambda x: x)
        
        # He initialization
        self.conv1.weights.data = np.random.normal(0, np.sqrt(2.0 / 25.0), size=self.conv1.weights.data.shape)
        self.conv1.bias.data = np.zeros(self.conv1.bias.data.shape)
        self.conv2.weights.data = np.random.normal(0, np.sqrt(2.0 / 150.0), size=self.conv2.weights.data.shape)
        self.conv2.bias.data = np.zeros(self.conv2.bias.data.shape)
        self.fc1.weights.data = np.random.normal(0, np.sqrt(2.0 / 400.0), size=self.fc1.weights.data.shape)
        self.fc1.bias.data = np.zeros(self.fc1.bias.data.shape)
        self.fc2.weights.data = np.random.normal(0, np.sqrt(2.0 / 120.0), size=self.fc2.weights.data.shape)
        self.fc2.bias.data = np.zeros(self.fc2.bias.data.shape)
        self.fc3.weights.data = np.random.normal(0, np.sqrt(1.0 / 84.0), size=self.fc3.weights.data.shape)
        self.fc3.bias.data = np.zeros(self.fc3.bias.data.shape)
        
    def __call__(self, x: Tensor) -> Tensor:
        x = self.pool1(relu(self.conv1(x)))
        x = self.pool2(relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x
        
    def params(self) -> list:
        return (self.conv1.model_params() +
                self.conv2.model_params() +
                [self.fc1.weights, self.fc1.bias,
                 self.fc2.weights, self.fc2.bias,
                 self.fc3.weights, self.fc3.bias])

model = LeNet5()
optimizer = Adam(model.params(), learning_rate=0.003)
criterion = SoftmaxCrossEntropy()

print("Training model...")
for epoch in range(5):
    for b in range(8):
        start_idx = b * 64
        end_idx = start_idx + 64
        x_batch = Tensor(X_train[start_idx:end_idx])
        y_batch = Tensor(y_train_oh[start_idx:end_idx])
        
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
print("Model trained!")

## 2. Visualizing Convolutional Filters (Kernels)

Let's extract and plot the 6 kernels of size $5\times5$ from the first convolutional layer (`conv1`). These kernels detect simple features (like vertical, horizontal, or diagonal edges) on the input pixels.

In [ ]:
# conv1.weights.data shape is (6, 1, 25) -> 6 filters, 1 input channel, 5x5 spatial size
kernels = model.conv1.weights.data.reshape(6, 5, 5)

fig, axes = plt.subplots(1, 6, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(kernels[i], cmap='bwr', vmin=-0.5, vmax=0.5)
    ax.set_title(f"Kernel {i}", fontsize=10, fontweight='bold')
    ax.axis('off')
plt.suptitle("Learned 5x5 conv1 Kernels (Blue=Negative, Red=Positive)", fontsize=12, fontweight='bold', y=1.05)
plt.show()

## 3. Visualizing Intermediate Feature Maps

Let's select a test image (a digit 5) and pass it through the network, plotting the outputs of each intermediate layer.

In [ ]:
# Find a digit '5' or '0' in the test set
img_idx = 0
while y_test_raw[img_idx] != 5:
    img_idx += 1

sample_image = X_test[img_idx:img_idx+1]
sample_tensor = Tensor(sample_image)

# Forward through layers explicitly to inspect activations
h_conv1 = model.conv1(sample_tensor)
h_relu1 = relu(h_conv1)
h_pool1 = model.pool1(h_relu1)

h_conv2 = model.conv2(h_pool1)
h_relu2 = relu(h_conv2)
h_pool2 = model.pool2(h_relu2)

# Plot original
plt.figure(figsize=(3, 3))
plt.imshow(sample_image[0, 0], cmap='gray')
plt.title(f"Input Image (Digit {y_test_raw[img_idx]})")
plt.axis('off')
plt.show()

### Feature Maps of Layer 1 (Conv1 and Pool1)
Let's plot the 6 channel outputs of the first convolutional block. Notice how different filters highlight horizontal vs. vertical contours of the digit.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    # Conv1 activations
    axes[0, i].imshow(h_conv1.data[0, i], cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title(f"Conv1 Ch {i}", fontsize=9)
    
    # Pool1 activations (downsampled)
    axes[1, i].imshow(h_pool1.data[0, i], cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title(f"Pool1 Ch {i}", fontsize=9)

plt.suptitle("Feature Maps after Conv1 (Top) and MaxPool1 (Bottom)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Feature Maps of Layer 2 (Conv2 and Pool2)
As we go deeper, the feature maps become smaller ($5\times5$ at Pool2) and represent higher-level, more abstract combinations of shapes.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for i in range(16):
    row = i // 8
    col = i % 8
    axes[row, col].imshow(h_pool2.data[0, i], cmap='gray')
    axes[row, col].axis('off')
    axes[row, col].set_title(f"Pool2 Ch {i}", fontsize=9)
plt.suptitle("Feature Maps after MaxPool2 (Deep representations of size 5x5)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Class Activation Maximization ("Dreams")

Now we will perform **Class Activation Maximization**. Instead of updating weights to fit images, we will keep the network parameters constant and perform **gradient ascent on a random noise image** to synthesize an input that maximizes the prediction score of a chosen target class (e.g., class 5 or 8).

This reveals what geometric shapes and templates the network is looking for.

In [ ]:
target_digit = 8

# Start with a neutral, slightly noisy background image
x_noise_np = np.random.uniform(0.4, 0.6, size=(1, 1, 28, 28))
x_dream = Tensor(x_noise_np)

lr_dream = 0.2
steps = 80

print(f"Optimizing input pixels to synthesize class '{target_digit}'...")
for step in range(steps):
    # Forward pass
    logits = model(x_dream)
    
    # Loss: we want to maximize logits[0, target_digit].
    # To maximize using gradient descent, we minimize negative logit value:
    loss_mask = np.zeros((1, 10))
    loss_mask[0, target_digit] = -1.0
    
    loss = (logits * Tensor(loss_mask)).sum()
    
    # Reset gradient of input pixels
    x_dream.grad = np.zeros(x_dream.shape())
    loss.backward()
    
    # Gradient ascent update on input pixels
    x_dream.data -= lr_dream * x_dream.grad
    # Clip to keep pixels in valid range [0, 1]
    x_dream.data = np.clip(x_dream.data, 0.0, 1.0)

# Plot the final synthesized "dream" image
plt.figure(figsize=(4, 4))
plt.imshow(x_dream.data[0, 0], cmap='inferno')
plt.title(f"Synthesized Class '{target_digit}' Dream", fontsize=12, fontweight='bold')
plt.axis('off')
plt.colorbar(label='Pixel Intensity')
plt.show()